# 03 — Verify the `cellflow` env on a GPU node
Executed inside `srun -p interactive --gres=gpu:1 -c 8 --mem=64G -t 2:00:00` (launched from max-login),
with kernel `cellflow`.

In [1]:
import os, sys, socket, subprocess
print("python:", sys.executable)
print("host:", socket.gethostname(), "| SLURM_JOB_ID:", os.environ.get("SLURM_JOB_ID"))
print("XLA_PYTHON_CLIENT_PREALLOCATE =", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))
print("HF_HOME =", os.environ.get("HF_HOME"))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv"], capture_output=True, text=True).stdout)

python: /fast/AG_Bunina/Pavel_1/miniforge3/envs/cellflow/bin/python
host: max027.mdc-berlin.net | SLURM_JOB_ID: 3369017
XLA_PYTHON_CLIENT_PREALLOCATE = false
HF_HOME = /fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache
name, memory.total [MiB], driver_version, compute_cap
NVIDIA A40, 46068 MiB, 610.57.04, 8.6



In [2]:
!{sys.executable} -c "import jax; print(jax.devices())"

[CudaDevice(id=0)]


In [3]:
# literal check from the task spec — cellflow-tools 0.0.9 does not define `cellflow.__version__` (AttributeError expected)
!{sys.executable} -c "import cellflow; print(cellflow.__version__)"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
AttributeError: module 'cellflow' has no attribute '__version__'


In [4]:
!{sys.executable} -c "import cellflow; from importlib.metadata import version; print('cellflow-tools', version('cellflow-tools'))"

cellflow-tools 0.0.9


In [5]:
import jax, jax.numpy as jnp
devs = jax.devices()
assert any(d.platform == "gpu" for d in devs), f"No GPU visible to JAX: {devs}"
x = jnp.ones((4096, 4096))
print("matmul on", (x @ x).devices(), "sum =", float((x @ x).sum()))
print("jax", jax.__version__, "| default backend:", jax.default_backend())

matmul on {CudaDevice(id=0)} sum = 68719476736.0
jax 0.10.2 | default backend: gpu
